# pytorch_nnfs
Let's learn some object oriented programming!

Using pytorch specifically for gpu acceleration.

In [1]:
import pandas
import torch
import matplotlib

# initialising torch accelerators
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
torch.set_default_device(device)
print(f"Using {device} device")

g = torch.Generator(device=device)

Using mps device


In [2]:
# data
data = torch.from_numpy(pandas.read_csv('../../data/train.csv').to_numpy())

def one_hot(data, outputs):
    return torch.eye(outputs)[Y_train]

# data preparation
X_train = data[:32000,1:].float()/255
Y_train = data[:32000,0]
Y_train_one_hot = one_hot(Y_train, 10)

X_val = data[32000:36000,1:].float()/255
Y_val = data[32000:36000,0]

X_test = data[36000:42000,1:].float()/255
Y_test = data[36000:42000,0]

# move data to the gpu
X_train = X_train.to(device)
Y_train = Y_train.to(device)
Y_train_one_hot = Y_train_one_hot.to(device)

# Todo:
Implement the new layer functions

In [ ]:
# initialisation functions
class init:
    class weight_init:
        @staticmethod
        def input_initialisation(neurons, device=device):
            return torch.ones(neurons, 1, device=device)

        def output_initialisation(neurons, device=device):
            return torch.ones(neurons, 1, device=device)

        @staticmethod
        def He_initialisation(neurons, num_weights, device=device):
            return torch.randn(neurons, num_weights, device=device) * torch.sqrt(2 / torch.tensor(num_weights))

    class bias_init:
        @staticmethod
        def zero_initialisation(neurons, device):
            return torch.zeros(neurons, 1, device=device)
    

init_dict = {
    'input': init.weight_init.input_initialisation,
    'output': init.weight_init.output_initialisation,
    'he': init.weight_init.He_initialisation,
}

bias_dict = {
    'zero': init.bias_init.zero_initialisation,
}

In [5]:
# Layer class
class Layer:
    def __init__(self, layer_type, device=device):
        self.layer_type=layer_type
        self.device=device
    def __repr__(self):
        return str(self.layer_type)

    def forward(self, data):
        raise NotImplementedError
    def backward(self, delta):
        raise NotImplementedError

"""
We are fixing this right now :)
"""

class Linear(Layer):
    def __init__(self, num_neurons, num_weights, weight_init, bias_init):
        super().__init__(layer_type='Linear')

        self.weights = init_dict[weight_init](num_neurons, num_weights, device=self.device)
        self.bias = bias_dict[bias_init](num_neurons, device=self.device)

    def forward(self, data):
        self.inputs = data
        self.pre_activations = self.weights @ data + self.bias
        return self.pre_activations

    def backward(self, delta):
        self.dW = delta @ self.inputs.T
        self.db = torch.sum(delta, dim=1, keepdim=True)
        delta_prev = self.weights.T @ delta
        return delta_prev

class LinearSoftmax(Layer): # not implemented yet
    def __init__(self, num_neurons, num_weights, weight_init, bias_init):
        super().__init__(layer_type='LinearSoftmax')

        self.weights = init_dict[weight_init](num_neurons, num_weights, device=self.device)
        self.bias = bias_dict[bias_init](num_neurons, device=self.device)

    def forward(self, data):
        self.inputs = data
        self.pre_activations = self.weights @ data + self.bias
        max_pre_activations = torch.max(self.pre_activations, dim=0, keepdim=True).values
        exp_pre_activations = torch.exp(self.pre_activations - max_pre_activations) # Z - max_Z to make sure we don't have an exploding number. maybe that's why it's called exp?
        sum_exp_pre_activations = torch.sum(exp_pre_activations, dim=0, keepdim=True)
        return exp_pre_activations / sum_exp_pre_activations

    def backward(self, outputs, Y_one_hot):
        delta = outputs - Y_one_hot #This is something that I should probably improve
        self.dW = delta @ self.inputs.T
        self.db = torch.sum(delta, dim=1, keepdim=True)
        delta_prev = self.weights.T @ delta
        return delta_prev

class ReLU(Layer):
    def __init__(self):
        super().__init__(layer_type='ReLU')

    def forward(self, data):
        self.pre_activations = data
        self.activations = torch.clamp(self.pre_activations, min=0)
        return self.activations

    def backward(self, delta):
        local_grad = (self.pre_activations > 0).float()
        delta_prev = delta * local_grad
        return delta_prev

layer1 = Linear(num_neurons=100, num_weights=784, weight_init='he', bias_init='zero')

In [8]:
class Sequential:
    def __init__(self, layer_config):
        self.layers=[]
        for layer in layer_config:
            layer_type = layer.pop('layer_type')

            # layers
            if layer_type == 'Linear':
                self.layers.append(Linear(**layer))
            if layer_type == 'LinearSoftmax':
                self.layers.append(LinearSoftmax(**layer))

            # activation functions
            if layer_type == 'ReLU':
                self.layers.append(ReLU(**layer))

    def __repr__(self):
        return str(self.layers)

    def forward(self, data):
        outputs = []

        outputs.append(self.layers[0].forward(data))
        for i in range(1, len(self.layers)):
            outputs.append(self.layers[i].forward(outputs[i-1]))
        print(outputs)

layer_config = [
    {'layer_type': 'Linear',        #0
     'num_neurons': 128,
     'num_weights': 784,
     'weight_init': 'he',
     'bias_init': 'zero',},         #1
    {'layer_type': 'ReLU'},         #2
    {'layer_type': 'Linear',        #3
     'num_neurons': 32,
     'num_weights': 128,
     'weight_init': 'he',
     'bias_init': 'zero'},
    {'layer_type': 'ReLU'},         #4
    {'layer_type': 'LinearSoftmax', #5
     'num_neurons': 10,
     'num_weights': 32,
     'weight_init': 'he',
     'bias_init': 'zero'},
]

NN = Sequential(layer_config)
NN.forward(X_train[:10].T)

[tensor([[-0.3502, -0.8295, -0.1753,  ..., -0.5400, -0.5330, -1.0318],
        [ 0.0510,  0.8485, -0.0702,  ...,  0.4012,  0.0317,  0.2345],
        [-0.2106, -0.5878,  0.2974,  ...,  0.1013,  0.4548, -0.0681],
        ...,
        [ 0.3861,  0.3490,  0.6027,  ...,  0.3574,  0.3574,  0.3285],
        [-0.3073, -0.3960, -0.1635,  ...,  0.2125,  0.0173, -0.5800],
        [-0.4163, -0.8879, -0.0783,  ..., -0.7301, -0.3002, -0.6455]],
       device='mps:0'), tensor([[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
        [0.0510, 0.8485, 0.0000,  ..., 0.4012, 0.0317, 0.2345],
        [0.0000, 0.0000, 0.2974,  ..., 0.1013, 0.4548, 0.0000],
        ...,
        [0.3861, 0.3490, 0.6027,  ..., 0.3574, 0.3574, 0.3285],
        [0.0000, 0.0000, 0.0000,  ..., 0.2125, 0.0173, 0.0000],
        [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],
       device='mps:0'), tensor([[-0.5773, -0.6010,  0.0855, -0.3823, -0.5958, -0.6932, -0.4271, -0.3650,
         -0.4475, -0.5025],
        [ 